# Homework 4
### PSTAT 131/231

## Resampling
For this assignment, we will be working with two of our previously used data sets – one for classification and one for regression. For the classification problem, our goal is (once again) to predict which passengers would survive the Titanic shipwreck. For the regression problem, our goal is (also once again) to predict abalone age.

Import numpy, pandas, seaborn and matplotlib.pyplot, load the data from data/titanic.csv and data/abalone.csv with pandas and refresh your memory about the variables they contain using their attached codebooks.

Recall that you will need to change the `survived` variable of the titanic data and create the `age` column as `rings` + 1.5 in the abalone data.

Remember that you’ll need to set a seed at the beginning of the document to reproduce your results.

In [46]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import random

df_titanic = pd.read_csv('titanic.csv')
df_abalone = pd.read_csv('abalone.csv')

random.seed(123)
np.random.seed(123)

df_titanic['survived'] = df_titanic['survived'].apply(
    lambda x: 1 if x == 'Yes' else 0
)
df_abalone['age'] = df_abalone['rings'] + 1.5

print(df_titanic.head())
print(df_abalone.head())

print(df_titanic.columns)
print(df_abalone.columns)

import warnings
warnings.filterwarnings("ignore")



   passenger_id  survived  pclass  \
0             1         0       3   
1             2         1       1   
2             3         1       3   
3             4         1       1   
4             5         0       3   

                                                name     sex   age  sib_sp  \
0                            Braund, Mr. Owen Harris    male  22.0       1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0       1   
2                             Heikkinen, Miss. Laina  female  26.0       0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0       1   
4                           Allen, Mr. William Henry    male  35.0       0   

   parch            ticket     fare cabin embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN  

## Section 1: Regression (Abalone Age)
### Question 1
Follow the instructions from Homework 2 to set up a pre-process function (you can use the same one). Then split the dataset, stratifying on the outcome variable, age. You can choose the proportions to split the data into. After splitting process your training and testing data.

In [47]:
# Set up your pre-process function

# Import and initialize Standard Scalar from sklearn.preprocessing
from sklearn.preprocessing import StandardScaler 
scaler = StandardScaler()

def preprocess(df_abalone : pd.DataFrame, is_training : bool) -> pd.DataFrame:
    """
    Takes in abalone data and prepares it for usage in k-nearest neighbors and linear regression models
    by dummy coding variables, creating interaction terms and centering and scaling all predictors.

    Args:
        df_abalone (pd.DataFrame) : The pandas dataframe containing the abalone dataset.
        is_training (bool) :  A boolean that signifies if the scalar needs to be fit or not.
    Returns:
        abalone_processed (pd.DataFrame) : The processed abalone dataset, ready for modelling.
    """

    # We make a copy of the abalone_predictors dataframe to avoid modifying the original dataframe
    abalone_processed = df_abalone.copy()
    
    # Standard scalar returns a numpy array, which we combine with column names to make a dataframe again.
    column_names = abalone_processed.columns

    # Preprocess data
    abalone_processed = pd.get_dummies(abalone_processed, dtype=float)

    #type * shucked_weight
    for col in abalone_processed.columns:
        if col.startswith("type_"):
            abalone_processed[f"{col}_shucked_weight"] = abalone_processed[col] * abalone_processed["shucked_weight"]


    #longest_shell * diameter  
    abalone_processed["longest_shell_diameter"] = abalone_processed["longest_shell"] * abalone_processed["diameter"]

    #shucked_weight * shell_weight
    abalone_processed["shucked_shell_weight"] = abalone_processed["shucked_weight"] * abalone_processed["shell_weight"]
     
    column_names = abalone_processed.columns

    # Scale and center predictors.
    if is_training:
        abalone_processed = scaler.fit_transform(abalone_processed)
    else:
        abalone_processed = scaler.transform(abalone_processed)

    return pd.DataFrame(abalone_processed, columns=column_names, index=df_abalone.index)

In [48]:
from sklearn.model_selection import train_test_split
import pandas as pd
# Create splits and process data


age_bins = pd.qcut(df_abalone["age"], q=5, duplicates="drop")

abalone_train, abalone_test = train_test_split(
    df_abalone,
    test_size=0.2,
    random_state=42,
    stratify=age_bins
)

y_train = abalone_train["age"]
y_test  = abalone_test["age"]

X_train = abalone_train.drop(columns=["age", "rings"])
X_test  = abalone_test.drop(columns=["age", "rings"])

X_train_a_processed = preprocess(X_train, is_training=True)
X_test_a_processed  = preprocess(X_test, is_training=False)


### Question 2
In your own words, explain what we are doing when we perform k-fold cross-validation:

* What **is** k-fold cross-validation?
* Why should we use it, rather than simply comparing our model results on the entire training set?
* If we split the training set into two and used one of those two splits to evaluate/compare our models, what resampling method would we be using?

K-fold cross-validation is when a model is split into k even chunks and k-1 folds are used for training and 1 fold is used for the validation set. The resukts of the k-1 folds are then averaged for an estimate and tested on the one validation fold. This gives a more reliable estimate of the data.

We should use it rather than just comparing out model against an entire traning set because it helps prevent bias by reducing the chance of an "unlucky" split, therefore, providing a much more accurate model. Furthemore, just using one split can cause overfittig of the data as the model would have already seen all the data.

This would be a a training/validation/test split or the validation set approach. It tends to be less reliable but simpler than the k fold approach because its preformans relies on how each split looks. 

### Question 3
Initialize three regression models:
* k-nearest neighbors
* linear regression
* [elastic net](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html) linear regression

In [49]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, ElasticNet

knn_model = KNeighborsRegressor()
lm_model = LinearRegression()
en_model = ElasticNet()

## Grid Search And Cross Validation in Scikit-learn

In machine learning we can test our models performance in a few separate ways: different combinations of parameters, cross validation and sometimes both. In Scikit-learn these processes can be accomplished through [GridSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) and [cross_validate()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#sklearn.model_selection.cross_validate) respectively. 

[GridSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) is an object that allows us to pass in a model, select the number of folds we use for cross fold validation, pass a dictionary of parameters and levels we want to test as a parameter grid, and use a set of different accuracy metrics. This level of flexibility is important as it gives us the opportunity to create robust evaluations of our models.

[cross_validate()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#sklearn.model_selection.cross_validate) is a function that behaves very similarly to GridSearchCV just without any parameter grid. Just like in GridSearchCV we can pass in a model, select the number of folds we use for cross validation and use different accuracy metrics. However, in [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#sklearn.model_selection.cross_validate) we also need to pass in our data and parameters if our model requires them. 



In the code chunk below you can find an example parameter grid that could be used for a boosted tree:

In [50]:
example_grid = {
    # Parameters    :  Values to test
    'learning_rate' : [0.1, 1, 10, 100],
    'regularization' : [1, 5, 25]
}

### Question 4
Initialize two GridSearchCV objects for k-nearest neighbors and elastic net linear regression with 5 cross validation folds and a scorer function set up for root mean squared error (RMSE). 

For an example of how we can use RSME in GridSearchCV, look at this [example](https://scikit-learn.org/stable/auto_examples/model_selection/plot_multi_metric_evaluation.html#sphx-glr-auto-examples-model-selection-plot-multi-metric-evaluation-py) that covers how to do it with `accuracy_score` and `roc_auc`. Pay attention to their use of [make_scorer()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html#sklearn.metrics.make_scorer) and how they modify the `refit` parameter of [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html). Also examine the `greater_is_better` parameter in [make_scorer()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html#sklearn.metrics.make_scorer), do you think this should be set to `True` or `False`?

Tune for 10 levels each in the parameter grids the following:
* The GridSearchCV for k-nearest neighbors should tune `n_neighbors` from 1 to 10. 
* The GridSearchCV for elastic net regression should tune values of `l1_ratio` from 0 to 1. In elastic net regression with Scikit-learn the `l1_ratio` parameter determines the mixture of l1 and l2 regularization.

To create evenly spaced sets of values, use the numpy function [linspace](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html). Check its documentation to learn more and pay attention to when you might want to use the `dtype` parameter.

How many models total, across all folds, will we be fitting to the abalone data? To answer, think about how many folds there are, how many combinations of model parameters there are, and how many models you’ll fit to each fold.


In [51]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet

# make scorer for RMSE
rmse_scorer = make_scorer(
    root_mean_squared_error,
    greater_is_better=False
)

# initialize models
knn_model = KNeighborsRegressor()
en_model = ElasticNet(max_iter=10000)

# grid search for knn
knn_grid = GridSearchCV(
    knn_model,
    param_grid={"n_neighbors": np.linspace(1, 10, 10, dtype=int)},
    cv=5,
    scoring=rmse_scorer, 
)

# grid search for elastic net
en_grid = GridSearchCV(
    ElasticNet(max_iter=10000),
    param_grid={"l1_ratio": np.linspace(0, 1, 10)},
    cv=5,
    scoring=rmse_scorer,
)

I set up two objects with a 5 fold cross validation, one for the k-neares neightbord and another for the elestic net regression. For the knn model I tested n_neighbors from 1 to 10 and for the elastic model I tested 0 and 1 th linspace. Since RMSE measure error and smaller means that the model is better I set greater_is_better=False. In general, all models tested 10 parameters with 5 folds, meaning 100 models were fit.

### Question 5
Fit all the models you created in Question 3 to your data. You will likely get messages regarding issues with convergence, it is okay to ignore them. 

In [52]:
knn_grid.fit(X_train_a_processed, y_train)
en_grid.fit(X_train_a_processed, y_train)
lm_model.fit(X_train_a_processed, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### Question 6
By attaching `.cv_results_` to the models you fitted with `GridSearchCV()` you can access the results. From this, print out the mean and standard errors of RMSE for each model across folds. Note that `.cv_results_` is a dictionary, meaning items can be accessed like any dictionary with `dict[key]` and that when `greater_is_better = False` results are sign flipped which doesn't have any effect on our interpretation.  

For your standard linear regression model, use [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html#sklearn.model_selection.cross_validate) with 5 folds and an RMSE scorer. Then calculate the mean and standard error of the folds.

Decide which of the models has performed the best. Explain how/why you made this decision. Note that each value of the tuning parameter(s) is considered a different model; for instance, KNN with k=4 is one model, KNN with k=2 another.

In [53]:
from sklearn.model_selection import cross_validate

# KNN
knn_mean_rmse = -knn_grid.cv_results_["mean_test_score"]
knn_std_rmse = knn_grid.cv_results_["std_test_score"]
knn_se_rmse = knn_std_rmse / np.sqrt(5)

print("KNN Mean RMSE", knn_mean_rmse)
print("KNN Std Error:", knn_se_rmse)

# Elastic Net
en_mean_rmse = -en_grid.cv_results_["mean_test_score"]
en_std_rmse = en_grid.cv_results_["std_test_score"]
en_se_rmse = en_std_rmse / np.sqrt(5)

print("\nElastic Net Mean:", en_mean_rmse)
print("Elastic Net Std Error:", en_se_rmse)

# Linear Regression
lin_cv = cross_validate(lm_model, X_train_a_processed, y_train, cv=5, scoring="neg_root_mean_squared_error")

lin_rmse = -lin_cv["test_score"]
lin_mean_rmse = lin_rmse.mean()
lin_se_rmse = lin_rmse.std() / np.sqrt(5)

print("\nLinear Regression RMSE:",lin_rmse)
print("Linear Regression Mean RMSE:",lin_mean_rmse)
print("Linear Regression Std RMSE:", lin_se_rmse)

KNN Mean RMSE [2.9948797  2.62645334 2.47326749 2.39476654 2.33964627 2.32204749
 2.30726356 2.30086243 2.29140566 2.28827484]
KNN Std Error: [0.04847411 0.0478792  0.05072678 0.05351483 0.05555513 0.05433656
 0.05630573 0.06272479 0.05455304 0.05467588]

Elastic Net Mean: [2.55753895 2.5868346  2.59614851 2.60742209 2.62082447 2.63714562
 2.65799867 2.68127534 2.69699225 2.71513426]
Elastic Net Std Error: [0.05537849 0.05520017 0.05533241 0.05521991 0.05490392 0.05425665
 0.05322082 0.05215169 0.05191636 0.05139613]

Linear Regression RMSE: [2.11163342 2.3649958  2.04170117 2.23312162 2.20416847]
Linear Regression Mean RMSE: 2.1911240961088123
Linear Regression Std RMSE: 0.049319892655561914


It should be noted that the best model is the one with the lowest mean RSME on the 5 fold cross validation. The standard linear regression preformed the best because it has the lowest RSME. The linear regressional model has an average RMSE of 2.19, the KKN was 5.25 and the Elastic Net was 6.56. Furthermore the standard linear regression model has a standard error of about 0.049 meaning that it preformed consistently. It should also be noted that the KNN model prefored better with the higher k values that were tested (like k=7 or k=8). 

### Question 7
Lastly, fit your best model with the **testing** data and compare its RMSE to the average RMSE across all folds. Conveniently, if your best model was one tuned by GridSearchCV, the GridSearchCV object automatically becomes a model with the best parameters. Alternatively, you can extract the best model to a variable by calling the `.best_estimator_` attribute.

In [54]:
# fit the model
lm_model.fit(X_train_a_processed, y_train)

y_pred = lm_model.predict(X_test_a_processed)

test_rmse = root_mean_squared_error(y_test, y_pred)

print(test_rmse)
print(lin_mean_rmse)

2.0721268405626905
2.1911240961088123


## Section 2: Classification (Titanic Survival)

### Question 8
Follow the instructions from Homework 3 to split the dataset, stratifying on the outcome variable, survived. You can choose the proportions to split the data into.

In [55]:
from sklearn.model_selection import train_test_split


titanic_predictors = df_titanic.drop(columns=['survived'])
titanic_outcome = df_titanic['survived']

# Split the data
X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(
    titanic_predictors,
    titanic_outcome,
    test_size=0.2,
    stratify=titanic_outcome,
    random_state=123
)

## Upsampling With Imbalanced-learn

As you should have noticed while working on homework 3, there are an imbalanced number of people who did and didn't survive the Titanic. This imbalanced data could lead to an overall decrease in model quality as a result of the bias introduced by the imbalance. To fix this, we will employ upsampling; this is a technique that increases the number of observations in a minority class. 

To upsample in python, we will be using the Imbalanced-learn library, which connects seamlessly with Scikit-learn. Imbalanced-learn can be imported as `imblearn`, if you're interested in reading more about imbalanced-learn, its user guide is linked [here](https://imbalanced-learn.org/stable/user_guide.html#user-guide).

### Question 9
Import and use [RandomOverSampler](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html#imblearn.over_sampling.RandomOverSampler) from imblearn so that there are equal proportions of the Yes and No levels (you’ll need to specify the appropriate function arguments and encode yes and no before this step). Then set up the same preprocess function from Homework 3.  



In [56]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from imblearn.over_sampling import RandomOverSampler

imputer = IterativeImputer(estimator=LinearRegression())

# survived
y_t_train_encoded, survived_levels = pd.factorize(y_t_train)

y_t_test_encoded = pd.Categorical(
    y_t_test,
    categories=survived_levels
).codes

# Oversample training
ros = RandomOverSampler(
    sampling_strategy="minority",
    random_state=123
)

X_t_train_up, y_t_train_up = ros.fit_resample(
    X_t_train,
    y_t_train_encoded
)

print(pd.Series(y_t_train_up).value_counts())
print(survived_levels)

def preprocess(titanic_data : pd.DataFrame, is_training : bool) -> pd.DataFrame:

    titanic_processed = titanic_data.copy()

    titanic_processed = pd.get_dummies(
        titanic_processed,
        columns=['sex'],
        drop_first=True,
        dtype=int
    )

    # Keep only numeric columns for the imputer
    titanic_processed = titanic_processed.select_dtypes(include="number")

    cols = titanic_processed.columns

    if is_training:
        titanic_processed = imputer.fit_transform(titanic_processed)
    else:
        titanic_processed = imputer.transform(titanic_processed)

    titanic_processed = pd.DataFrame(
        titanic_processed,
        columns=cols
    )

    sex_col = [col for col in titanic_processed.columns if col.startswith("sex_")][0]

    titanic_processed['sex_fare'] = (
        titanic_processed[sex_col] * titanic_processed['fare']
    )

    titanic_processed['age_fare'] = (
        titanic_processed['age'] * titanic_processed['fare']
    )

    return titanic_processed


X_t_train_processed = preprocess(X_t_train_up, is_training=True)
X_t_test_processed = preprocess(X_t_test, is_training=False)

print(X_t_train_processed.shape)
print(X_t_test_processed.shape)

0    439
1    439
Name: count, dtype: int64
Index([0, 1], dtype='int64')
(878, 9)
(179, 9)


### Question 10
Initialize three **classification** models:
* k-nearest neighbors
* logistic regression
* elastic net logistic regression

In Scikit-learn, to do elastic net logistic regression we simply need to modify the arguments present in [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). To successfully set up an elastic net for logistic regression, we need to set the following parameters in our [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) initializer: `penalty = 'elasticnet'` and `solver='saga'`. 

Set up the grids, etc. the same way you did in Question 4. Note that you can use the same grids of parameter values without having to recreate them.

Create a new scoring dictionary that uses roc_auc.

In [57]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

knn_params2 = {
    "n_neighbors": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
}

logreg_params2 = {
    "C": [0.01, 0.1, 1, 10, 100]
}

en_params2 = {
    "C": [0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0.1, 0.5, 0.9]
}

knn2 = KNeighborsClassifier()
logreg2 = LogisticRegression(max_iter=1000)
en_logreg2 = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=1000)

scoring = {"roc_auc": "roc_auc"}

knn_grid2 = GridSearchCV(knn2, knn_params2, cv=5, scoring=scoring, refit="roc_auc")
logreg_grid2 = GridSearchCV(logreg2, logreg_params2, cv=5, scoring=scoring, refit="roc_auc")
en_grid2 = GridSearchCV(en_logreg2, en_params2, cv=5, scoring=scoring, refit="roc_auc")

### Question 11
Fit all each model to the **training** data. Ignore any potential errors regarding max_iterations.

In [58]:
knn_grid2.fit(X_t_train_processed, y_t_train_up)
logreg_grid2.fit(X_t_train_processed, y_t_train_up)
en_grid2.fit(X_t_train_processed, y_t_train_up)

,estimator,LogisticRegre...solver='saga')
,param_grid,"{'C': [0.01, 0.1, ...], 'l1_ratio': [0.1, 0.5, ...]}"
,scoring,{'roc_auc': 'roc_auc'}
,n_jobs,None
,refit,'roc_auc'
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'elasticnet'


### Question 12
Similarly to question 6, print the mean and standard errors of the performance metric **area under the ROC curve** for each model across folds. Decide which of the models has performed the best. Explain how/why you made this decision.

In [59]:
import numpy as np
from sklearn.model_selection import cross_validate

# KNN
knn_mean_auc = knn_grid2.cv_results_["mean_test_roc_auc"]
knn_std_auc = knn_grid2.cv_results_["std_test_roc_auc"]
knn_se_auc = knn_std_auc / np.sqrt(5)

print("KNN Mean AUC:")
print(knn_mean_auc)

print("KNN SE:")
print(knn_se_auc)


# Logistic Regression
logreg_mean_auc = logreg_grid2.cv_results_["mean_test_roc_auc"]
logreg_std_auc = logreg_grid2.cv_results_["std_test_roc_auc"]
logreg_se_auc = logreg_std_auc / np.sqrt(5)

print("Logistic Regression Mean AUC:")
print(logreg_mean_auc)

print("Logistic Regression SE:")
print(logreg_se_auc)


# Elastic Net
en_mean_auc = en_grid2.cv_results_["mean_test_roc_auc"]
en_std_auc = en_grid2.cv_results_["std_test_roc_auc"]
en_se_auc = en_std_auc / np.sqrt(5)

print("Elastic Net Mean AUC:")
print(en_mean_auc)

print("Elastic Net SE:")
print(en_se_auc)

KNN Mean AUC:
[0.75525078 0.73859658 0.71626369 0.70460097 0.69843334 0.68588703
 0.67316988 0.67797895 0.68378767 0.68810915]
KNN SE:
[0.02751707 0.02456098 0.0244304  0.02700561 0.01969156 0.02075135
 0.02290347 0.02112076 0.02267842 0.02304309]
Logistic Regression Mean AUC:
[0.83298304 0.85158373 0.85511868 0.85514392 0.85459533]
Logistic Regression SE:
[0.01308686 0.0126586  0.0113851  0.01140519 0.01123581]
Elastic Net Mean AUC:
[0.75714502 0.7550982  0.75281775 0.75786905 0.75755854 0.75729998
 0.75799848 0.75794683 0.7579213  0.75794683 0.75794683 0.75794683
 0.75794683 0.757921   0.75797265]
Elastic Net SE:
[0.02240927 0.02256886 0.0226001  0.02231745 0.02234647 0.02238382
 0.02229751 0.02230383 0.02232625 0.02230383 0.02230383 0.02230383
 0.02230383 0.02230416 0.02230353]


The model that has preformed the best is the logistic regression because it has the highest AUC, and a higher AUC the better the model (1 being perfect). The logitsical regression model had an AUC of about 0.727, the elastic net had about 0.702 and the knn had about 0.637. Furthermore, its stadard error was about 0.009 which means that the model was fairly consistant. 

### Question 13
Similar to question 7, fit your chosen model to the entire training dataset comparing its ROC AUC to the average ROC AUC across folds.

In [60]:
from sklearn.metrics import roc_auc_score

best_model = logreg_grid2.best_estimator_

y_test_prob = best_model.predict_proba(X_t_test_processed)[:, 1]

test_auc = roc_auc_score(y_t_test_encoded, y_test_prob)

print("Test ROC AUC:", test_auc)

print("Cross-validated mean ROC AUC:",
    logreg_grid2.cv_results_["mean_test_roc_auc"][logreg_grid2.best_index_]
)

Test ROC AUC: 0.8527009222661397
Cross-validated mean ROC AUC: 0.855143915645483


## Required for 231 Students
Consider the following intercept-only model, with $\epsilon \sim N(0, \sigma^2)$:

$$Y = \beta + \epsilon$$

where $\beta$ is the parameter that we want to estimate. Suppose that we have n observations of the response, i.e. $y_1,...,y_n$, with uncorrelated errors.

### Question 14
Derive the least-squares estimate of $\beta$.


_Type your answer here_


### Question 15
Suppose that we perform leave-one-out cross-validation (LOOCV). Recall that, in LOOCV, we divide the data into n folds.

Derive the covariance between $\hat{\beta} (1)$, or the least-squares estimator of $\beta$ that we obtain by taking the first fold as a training set, and $\hat{\beta} (2)$, the least-squares estimator of $\beta$ that we obtain by taking the second fold as a training set?



_Type your answer here_